# Convolutional Neural Network (CNN) - Cats vs Dogs Classifier

This notebook builds a simple CNN using TensorFlow/Keras to classify images into two classes: **Cat** or **Dog**. It downloads a small ready-made dataset automatically, so no manual upload or Google Drive mounting is required — just run the cells in order.

### Importing the libraries

In [ ]:
import tensorflow as tffrom tensorflow.keras.preprocessing.image import ImageDataGenerator#Image preprocessing and Data augmentation

In [ ]:
tf.__version__

## Part 0 - Downloading the Dataset

We use TensorFlow's built-in **Cats vs Dogs (small)** dataset (a filtered subset of the Kaggle Cats vs Dogs dataset, ~2000 training + 1000 validation images). It downloads directly from Google's servers, so this works the same on Colab or on your own computer — no Kaggle account or Drive mount needed.

In [ ]:
import osimport zipfiledataset_url = 'https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip'zip_path = tf.keras.utils.get_file('cats_and_dogs_filtered.zip', origin=dataset_url, extract=True)base_dir = os.path.join(os.path.dirname(zip_path), 'cats_and_dogs_filtered')train_dir = os.path.join(base_dir, 'train')test_dir = os.path.join(base_dir, 'validation')print('Training folder :', train_dir)print('Test folder     :', test_dir)

## Part 1 - Data Preprocessing

### Preprocessing the Training set

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255,                                   shear_range = 0.2,                                   zoom_range = 0.2,                                   horizontal_flip = True)training_set = train_datagen.flow_from_directory(train_dir,                                                 target_size = (64, 64),                                                 batch_size = 32,                                                 class_mode = 'binary')

### Preprocessing the Test set

In [ ]:
test_datagen = ImageDataGenerator(rescale = 1./255)test_set = test_datagen.flow_from_directory(test_dir,                                            target_size = (64, 64),                                            batch_size = 32,                                            class_mode = 'binary')

## Part 2 - Building the CNN

### Initialising the CNN

In [ ]:
cnn = tf.keras.models.Sequential()

### Step 1 - Convolution

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

### Step 2 - Pooling

In [ ]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Adding a second convolutional layer

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Step 3 - Flattening

In [ ]:
cnn.add(tf.keras.layers.Flatten()) #Converts the 2D feature maps into a 1D vector for the dense layers.

### Step 4 - Full Connection

In [ ]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu')) #number of neurons=128

### Step 5 - Output Layer

In [ ]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

## Part 3 - Training the CNN

### Compiling the CNN

In [ ]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])# Optimizer – how the model updates its weights.# Loss Function – how the model measures prediction errors.# Metrics – how the model's performance is evaluated during training.

### Training the CNN on the Training set and evaluating it on the Test set

In [ ]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 10)

## Part 4 - Making a Single Prediction

This is the **application-oriented** part: pick any real cat/dog photo, feed it to the trained model, and it tells you whether it thinks it's a cat or a dog.

In [ ]:
import numpy as npfrom tensorflow.keras.preprocessing import image# Grab one real image straight out of the validation set so it always existssample_path = os.path.join(test_dir, 'cats', os.listdir(os.path.join(test_dir, 'cats'))[0])test_image = image.load_img(sample_path, target_size = (64, 64))test_image = image.img_to_array(test_image)test_image = np.expand_dims(test_image, axis = 0)result = cnn.predict(test_image)print(training_set.class_indices)if result[0][0] == 1:    prediction = 'dog'else:    prediction = 'cat'print('Image path :', sample_path)print('Prediction :', prediction)

### (Optional) Try your own image

If you're on Colab, upload any cat/dog photo and change `my_image_path` below to test the model on it.

In [ ]:
my_image_path = sample_path  # <-- replace with your own uploaded image pathmy_image = image.load_img(my_image_path, target_size = (64, 64))my_image_arr = image.img_to_array(my_image)my_image_arr = np.expand_dims(my_image_arr, axis = 0)my_result = cnn.predict(my_image_arr)my_prediction = 'dog' if my_result[0][0] == 1 else 'cat'print('Prediction:', my_prediction)